# Employer Brand Diagnosis: ING vs. Peers

**Dataset:** Glassdoor Job Reviews 1 & 2 — David Gauthier ([Kaggle](https://www.kaggle.com/datasets/davidgauthier/glassdoor-job-reviews)). Merged and filtered by the instructor into a single Parquet file (July 2018 - July 2023, firms with 300+ reviews).

**Objective:** Identify ING's strengths (to exploit in employer branding and recruitment) and improvement areas (the ones driving employees out), always compared against its main competitors. A problem is only a priority if ING is doing worse than the market.

**Approach:** Sentiment analysis (pre-trained transformer) as quality control, and topic modeling (BERTopic) to extract what employees actually talk about in their pros and cons.

**Peers:** Santander, BNP-Paribas, Deutsche-Bank and Lloyds-Banking-Group. All of them are large European banks with a retail business, so they compete with ING for similar profiles (technology, operations, risk, business). Deutsche-Bank is more investment-banking oriented, which I'll keep in mind when interpreting the results.


## 0. Imports & SetUp

In [ ]:
# Standard library
import re
import warnings
from pathlib import Path

# Dataframe management
import pandas as pd
import numpy as np

# Graphs
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

In [ ]:
# Project configuration
DATA_PATH = Path("data/glassdoor_reviews_hr.parquet")

TARGET = "ING"
PEERS = ["Santander", "BNP-Paribas", "Deutsche-Bank", "Lloyds-Banking-Group"]
FIRMS = [TARGET] + PEERS

# Cap per firm. ING has 1,681 reviews, so a 1,500 cap barely trims it, while giving every firm
# the same weight in the joint topic model (otherwise the biggest firm would dominate the topics)
MAX_REVIEWS_PER_FIRM = 1500
SEED = 42

# Plot style: ING highlighted, peers in neutral tones
sns.set_theme(style="whitegrid")
PALETTE = {TARGET: "#FF6200", "Santander": "#9E9E9E", "BNP-Paribas": "#757575",
           "Deutsche-Bank": "#BDBDBD", "Lloyds-Banking-Group": "#616161"}

In [ ]:
# Data import from local Parquet. Filtering at read time avoids loading the whole file
raw_df = pd.read_parquet(DATA_PATH, filters=[("firm", "in", FIRMS)])

# Firm names must match Glassdoor's exactly, so fail early if any is missing
missing_firms = set(FIRMS) - set(raw_df["firm"].unique())
assert not missing_firms, f"Firms not found in the Parquet: {missing_firms}"

## 1. EDA

### 1.1. DataFrame Quality Analysis

**DataFrame dimensions**

In [ ]:
print(f"Rows: {raw_df.shape[0]}")
print(f"Cols: {raw_df.shape[1]}")

**Col type**

In [ ]:
raw_df.info()

**Null & Duplicates Count**

In [ ]:
null_count = raw_df.isnull().sum()
dupl_count = raw_df.duplicated(["firm", "date_review", "pros", "cons"]).sum()

print("Nulls")
print(null_count[null_count > 0])
print("Duplicates: ", dupl_count)
print("\nDate range:", raw_df["date_review"].min(), "->", raw_df["date_review"].max())

Nulls will be handled after casting the columns. The instructor already removed duplicated reviews across the two source datasets, so I only check duplicates on `firm`, `date_review`, `pros` and `cons`, which are the columns that identify a review (the dataset has no review ID).

> **TODO:** if duplicates are not residual, check whether they are copy-pasted reviews before dropping them.

**Available reviews per firm**

In [ ]:
print(raw_df["firm"].value_counts())

### 1.2. Col Casting & Feature Engineering

`current` mixes employment status and tenure (e.g. _"Former Employee, more than 3 years"_), so I split it into two variables. `recommend` is translated from its code (v = yes, o = no opinion, x = no).

In [ ]:
# Auxiliar funcs. to split the `current` column
def get_status(s):
    s = str(s).lower()
    if s.startswith("current"):
        return "Current"
    if s.startswith("former"):
        return "Former"
    return "Unknown"

def get_tenure(s):
    m = re.search(r"(less|more) than (\d+) year", str(s))
    return f"{m.group(1)} than {m.group(2)}y" if m else "n/a"

raw_df["status"] = raw_df["current"].map(get_status)
raw_df["tenure"] = raw_df["current"].map(get_tenure)
raw_df["recommends"] = raw_df["recommend"].map({"v": "Yes", "o": "Neutral", "x": "No"})
raw_df["date_review"] = pd.to_datetime(raw_df["date_review"])
raw_df["year"] = raw_df["date_review"].dt.year

# Firm as ordered categorical, so every table and plot keeps ING first
raw_df["firm"] = pd.Categorical(raw_df["firm"], categories=FIRMS, ordered=True)

**Text cleaning**

In [ ]:
# Nulls are filled with "" BEFORE casting to str, otherwise they become the literal text "nan"
for col in ["pros", "cons"]:
    raw_df[col] = raw_df[col].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

### 1.3. Null Treatment

Only reviews with some text can be used in the NLP part, but the rating information of the rest is still valid for the EDA, so I check how many reviews are affected before deciding.

In [ ]:
print("Empty pros:", (raw_df["pros"] == "").sum())
print("Empty cons:", (raw_df["cons"] == "").sum())
print("Both empty:", ((raw_df["pros"] == "") & (raw_df["cons"] == "")).sum())
print("\nMissing recommend:", raw_df["recommends"].isna().sum())
print("Unknown status:", (raw_df["status"] == "Unknown").sum())

Reviews with both `pros` and `cons` empty have nothing to analyse, so I drop them. Reviews with only one of them empty are kept: they are still valid for the other text column, and the empty one is simply excluded from that model later on. Missing `recommends` are left as `NaN` (not imputed) and are excluded from the recommendation rate.

In [ ]:
clean_df = raw_df[~((raw_df["pros"] == "") & (raw_df["cons"] == ""))].copy()
print(f"Dropped rows: {len(raw_df) - len(clean_df)}")

### 1.4. Balanced Sampling

Using a fixed seed so the whole project is reproducible. Every firm is capped at `MAX_REVIEWS_PER_FIRM`, so no firm has more weight than the others in the joint topic model, and it also keeps computing time reasonable for the transformer models.

In [ ]:
sample_df = pd.concat(
    [g.sample(n=min(MAX_REVIEWS_PER_FIRM, len(g)), random_state=SEED)
     for _, g in clean_df.groupby("firm", observed=True)],
    ignore_index=True
)

print(pd.DataFrame({"available": clean_df["firm"].value_counts(), "sample": sample_df["firm"].value_counts()}).loc[FIRMS])
print(f"\nWorking sample: {len(sample_df)} reviews")

From now on every analysis uses `sample_df`.

### 1.5. Business focused analysis

**Note:** Numbers above bars represent the number of reviews, not the rating.

In [ ]:
# Auxiliar func. to plot mean rating with review counts
def barplot_with_counts(data, x, order, ax, title):
    sns.barplot(data=data, x=x, y="overall_rating", order=order, ax=ax, errorbar=None, color="#9E9E9E")
    counts = data[x].value_counts()
    for i, category in enumerate(order):
        n = counts.get(category, 0)
        height = data[data[x] == category]["overall_rating"].mean()
        ax.text(i, height + 0.05, f"{n}", ha="center", fontsize=9)
    ax.set_title(title)
    ax.set_ylim(0, 5)
    ax.set_ylabel("Mean rating (1-5)")

**Firm summary**

In [ ]:
# Recommendation rate is computed only over reviews with an opinion (Yes / No). "Neutral" = no opinion
with_opinion = sample_df[sample_df["recommends"].isin(["Yes", "No"])]

summary = pd.DataFrame({
    "reviews": sample_df["firm"].value_counts(),
    "mean_rating": sample_df.groupby("firm", observed=True)["overall_rating"].mean(),
    "pct_recommend": with_opinion.groupby("firm", observed=True)["recommends"].apply(lambda s: (s == "Yes").mean() * 100),
    "pct_current": sample_df.groupby("firm", observed=True)["status"].apply(lambda s: (s == "Current").mean() * 100),
}).loc[FIRMS].round(2)

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean rating with 95% CI, so small differences between firms are not over-interpreted
sns.pointplot(data=sample_df, x="firm", y="overall_rating", order=FIRMS, errorbar=("ci", 95),
              linestyle="none", color="#424242", ax=axes[0])
axes[0].set_title("Mean rating by firm (95% CI)")
axes[0].set_ylabel("Mean rating (1-5)")
axes[0].tick_params(axis="x", rotation=30)

# Recommendation rate
sns.barplot(x=summary.index, y=summary["pct_recommend"], hue=summary.index, palette=PALETTE, order=FIRMS, legend=False, ax=axes[1])
axes[1].set_title("% of employees who recommend the firm")
axes[1].set_ylabel("% recommend")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

> **TODO:** Is ING above or below the peers' average in rating and recommendation? Do the confidence intervals overlap? If they do, the difference is not reliable and I shouldn't present it as a finding.

**Rating ~ Employment status (current vs. former)**

In [ ]:
status_df = sample_df[sample_df["status"] != "Unknown"]

gap = status_df.pivot_table(index="firm", columns="status", values="overall_rating", aggfunc="mean", observed=True).loc[FIRMS]
gap["gap (Current - Former)"] = gap["Current"] - gap["Former"]
print(gap.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=status_df, x="firm", y="overall_rating", hue="status", order=FIRMS,
            hue_order=["Current", "Former"], errorbar=None, ax=ax)
ax.set_title("Mean rating: current vs. former employees")
ax.set_ylabel("Mean rating (1-5)")
ax.set_ylim(0, 5)
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

Former employees are expected to rate lower in every firm (selection bias: people who leave are, on average, less satisfied), so the absolute difference is not informative by itself. What matters is whether ING's gap is larger than its peers'.

> **TODO:** Compare ING's gap against the peers' gaps.

**Rating evolution**

In [ ]:
# Reviews per year and firm. 2018 and 2023 are partial years (July 2018 - July 2023)
print(sample_df.pivot_table(index="year", columns="firm", values="overall_rating", aggfunc="count", observed=True)[FIRMS])

fig, ax = plt.subplots(figsize=(10, 5))

for firm in FIRMS:
    firm_year = sample_df[sample_df["firm"] == firm].groupby("year")["overall_rating"].mean()
    ax.plot(firm_year.index, firm_year.values, marker="o", color=PALETTE[firm],
            lw=3.5 if firm == TARGET else 1.5, label=firm)

ax.set_title("Mean rating by year")
ax.set_ylabel("Mean rating (1-5)")
ax.set_xticks(sorted(sample_df["year"].unique()))
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()
plt.show()

> **TODO:** Does ING improve or worsen over time relative to its peers? Is there any inflection point (e.g. 2020)? If a year has few reviews, I won't interpret it.

**Rating ~ Recommendation**

In [ ]:
# If rating and recommendation tell the same story, rating should drop from Yes -> Neutral -> No
rec_order = ["Yes", "Neutral", "No"]

fig, ax = plt.subplots(figsize=(7, 5))
barplot_with_counts(sample_df.dropna(subset=["recommends"]), "recommends", rec_order, ax, "Mean rating by recommendation")

plt.tight_layout()
plt.show()

print(sample_df.pivot_table(index="firm", columns="recommends", values="overall_rating", aggfunc="mean", observed=True)
      .loc[FIRMS][rec_order].round(2))

This is a consistency check rather than an insight: if `overall_rating` and `recommend` disagreed, I couldn't use either of them later to validate the topics.

> **TODO:** Confirm the expected pattern holds in all five firms.

**Text length**

In [ ]:
sample_df["n_words_pros"] = sample_df["pros"].str.split().str.len()
sample_df["n_words_cons"] = sample_df["cons"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, ["n_words_pros", "n_words_cons"], ["Pros", "Cons"]):
    sns.boxplot(data=sample_df, x="firm", y=col, order=FIRMS, showfliers=False, showmeans=True,
                meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black"}, ax=ax)
    ax.set_title(f"{title}: words per review (diamond = mean)")
    ax.set_ylabel("Words")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Transformers truncate at 512 tokens (~350-400 words), so check how many reviews would be cut
for col in ["n_words_pros", "n_words_cons"]:
    print(f"{col}: {round((sample_df[col] > 350).mean() * 100, 2)}% of reviews above 350 words")

> **TODO:** If the share of reviews above 350 words is small, truncation is not a real problem for the sentiment model and I'll keep `truncation=True` as is.